# 01 · Descarga común

Este notebook explica y prueba la función `descargar()` de `src/descarga.py`, que usan **todos**
los notebooks del proyecto para pedir páginas a la web.

La función está en un fichero `.py` (y no aquí) para que cada notebook la importe sin copiarla:
si hay que cambiar algo (la pausa, el User-Agent...) se cambia en un solo sitio.

Buenas prácticas que aplica:

| Requisito | Cómo |
|---|---|
| User-Agent propio | `USER_AGENT` identifica el proyecto y un contacto |
| Pausa ≥ 1 s entre peticiones | `PAUSA = 1.5` s, solo entre peticiones reales |
| Guardar en bytes | `respuesta.content` se escribe tal cual en `datos/raw/` |
| No repetir descargas | si el fichero ya existe, se lee del disco |

> En este notebook **no se muestra el contenido** de las páginas: solo tamaños, rutas y tiempos.

## 1. Importar la función

In [1]:
import sys
import time
from pathlib import Path

# Los notebooks están en notebooks/; añadimos la raíz del repo para poder importar src/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import requests
from src.descarga import descargar, ruta_raw, USER_AGENT, PAUSA, DIR_RAW

print("User-Agent:", USER_AGENT)
print("Pausa entre peticiones:", PAUSA, "s")
print("Carpeta raw:", DIR_RAW.relative_to(RAIZ))

User-Agent: CorunaCorreBot/0.1 (proyecto educativo Big Data; contacto: raul.antelobdi@gmail.com)
Pausa entre peticiones: 1.5 s
Carpeta raw: datos\raw


## 2. Dónde se guarda cada URL

`ruta_raw()` convierte la URL en una ruta fija dentro de `datos/raw/`: la misma URL da siempre
el mismo fichero, así sabemos si ya está descargada.

In [2]:
for url in [
    "https://corunacorre.com/web/resultado/",
    "https://corunacorre.com/gl/web/resultado/evento-3216",
    "https://corunacorre.com/gl/web/resultado/evento-3216?page=2",
]:
    print(url, "->", ruta_raw(url).relative_to(RAIZ))

https://corunacorre.com/web/resultado/ -> datos\raw\corunacorre.com\web_resultado.html
https://corunacorre.com/gl/web/resultado/evento-3216 -> datos\raw\corunacorre.com\gl_web_resultado_evento-3216.html
https://corunacorre.com/gl/web/resultado/evento-3216?page=2 -> datos\raw\corunacorre.com\gl_web_resultado_evento-3216_page_2.html


## 3. Primera descarga

Descargamos la página de resultados. Si ya estaba en `datos/raw/` (por una ejecución anterior),
no se hace ninguna petición.

In [3]:
URL = "https://corunacorre.com/web/resultado/"
ruta = ruta_raw(URL)
ya_estaba = ruta.exists()

t0 = time.perf_counter()
contenido = descargar(URL)
t1 = time.perf_counter()

print("¿Ya estaba en disco?:", ya_estaba)
print("Tipo devuelto:", type(contenido).__name__)
print("Tamaño:", len(contenido), "bytes")
print("Guardado en:", ruta.relative_to(RAIZ))
print(f"Tiempo: {t1 - t0:.2f} s")

¿Ya estaba en disco?: False
Tipo devuelto: bytes
Tamaño: 60967 bytes
Guardado en: datos\raw\corunacorre.com\web_resultado.html
Tiempo: 0.46 s


## 4. Segunda llamada: no se repite la descarga

Al pedir la misma URL otra vez, la función lee el fichero del disco: no hay petición ni pausa,
así que tarda casi 0 s y devuelve exactamente los mismos bytes.

In [4]:
t0 = time.perf_counter()
otra_vez = descargar(URL)
t1 = time.perf_counter()

print("Mismos bytes:", otra_vez == contenido)
print(f"Tiempo: {t1 - t0:.4f} s")
assert otra_vez == contenido and (t1 - t0) < 0.5

Mismos bytes: True
Tiempo: 0.0400 s


## 5. La pausa entre peticiones

Dos peticiones reales seguidas: la segunda no empieza hasta que pasan `PAUSA` segundos desde
que empezó la primera. Medimos el instante en que **empieza** cada petición
(`descarga._ultima_peticion`, que la función actualiza justo antes de llamar a `requests`).

Usamos `forzar=True` para obligar a descargar aunque el fichero exista, y guardamos en una
carpeta de prueba que luego borramos.

In [5]:
from src import descarga

prueba = DIR_RAW / "_prueba"
urls = ["https://corunacorre.com/gl/web/nota/legal/", "https://corunacorre.com/gl/web/politica/privacidad/"]

inicios = []
for i, u in enumerate(urls):
    descargar(u, ruta=prueba / f"{i}.html", forzar=True)
    inicios.append(descarga._ultima_peticion)

separacion = inicios[1] - inicios[0]
print(f"Separación entre el inicio de las dos peticiones: {separacion:.2f} s (mínimo {PAUSA} s)")
assert separacion >= PAUSA

for f in prueba.glob("*"):
    f.unlink()
prueba.rmdir()

Separación entre el inicio de las dos peticiones: 1.50 s (mínimo 1.5 s)


## 6. Errores HTTP: el caso de robots.txt

Si la respuesta no es correcta, la función lanza `requests.HTTPError` y **no guarda nada**.
Lo comprobamos con `robots.txt`, que en esta web no existe (redirige a `/gl/robots.txt` y da 404).

In [6]:
URL_ROBOTS = "https://corunacorre.com/robots.txt"
try:
    descargar(URL_ROBOTS, forzar=True)
except requests.HTTPError as e:
    print("Error HTTP:", e.response.status_code, "en", e.response.url)
print("¿Se ha guardado algo?:", ruta_raw(URL_ROBOTS).exists())

Error HTTP: 404 en https://corunacorre.com/gl/robots.txt
¿Se ha guardado algo?: False


## Conclusión

- Todas las peticiones del proyecto se hacen con `descargar()`.
- Se identifica con su User-Agent, espera al menos 1,5 s entre peticiones y guarda los bytes en `datos/raw/`.
- Si la página ya está en disco, no vuelve a pedirla.